In [27]:
from fpl_project.scripts import config
from fpl_project.scripts.data_utils.features_config import FeaturesConfig
from fpl_project.scripts.model_utils.data import FPLDataPipe, train_test_split
from fpl_project.scripts.model_utils.trainer import set_seed
from fpl_project.scripts.model_utils.selector import ModelSelector, initialize_device

from typing import NamedTuple
from dataclasses import dataclass
from typing import Any, Dict
from fpl_project.scripts.model_utils.models import FPLLoss

In [28]:
import pandas as pd
import scipy.stats as stats

pd.set_option('display.max_columns', None)

from sklearn.ensemble import RandomForestRegressor
import glum
import xgboost
import catboost

import torch

import numpy as np
import seaborn as sns
sns.set_style("whitegrid")

import matplotlib.pyplot as plt


In [29]:
device = initialize_device()
device

'cpu'

In [30]:
set_seed(77)
if not config.FEATURES_ENGINEERING.exists():
    from fpl_project.scripts.data_utils.features_processing import build_and_save_features_engineering_data
    build_and_save_features_engineering_data()
data = pd.read_parquet(config.FEATURES_ENGINEERING)

In [31]:
data.total_points = data.total_points.clip(lower=0)

In [32]:
data.head()

,total_points,name,element,code,season,gw,kickoff_time,team,opponent_team,prob_win,prob_draw,prob_over_2.5,position,was_home,goals_conceded_ema_3,player_match_xa_expected_ema_3,transfers_trend_ema_3,clean_sheets_ema_3,transfers_in_ema_3,ict_index_ema_3,xa_ema_3,assists_ema_3,player_match_xg_expected_ema_3,hype_vs_odds_ratio_ema_3,xg_ema_3,value_ema_3,starts_ema_3,xg_per_90_ema_3,transfers_out_ema_3,xa_per_90_ema_3,transfers_balance_ema_3,team_h_score_ema_3,penalties_missed_ema_3,red_cards_ema_3,xg_conceded_per_90_ema_3,yellow_cards_ema_3,selected_ema_3,total_points_ema_3,penalties_saved_ema_3,saves_ema_3,threat_ema_3,xg_conceded_ema_3,own_goals_ema_3,xg_involvements_per_90_ema_3,team_a_score_ema_3,match_value_efficiency_ema_3,goals_scored_ema_3,creativity_ema_3,xp_ema_3,bonus_ema_3,goals_conceded_lagged_1,player_match_xa_expected_lagged_1,transfers_trend_lagged_1,clean_sheets_lagged_1,xa_lagged_1,assists_lagged_1,player_match_xg_expected_lagged_1,hype_vs_odds_ratio_lagged_1,xg_lagged_1,xg_per_90_lagged_1,xa_per_90_lagged_1,transfers_balance_lagged_1,team_h_score_lagged_1,penalties_missed_lagged_1,red_cards_lagged_1,xg_conceded_per_90_lagged_1,yellow_cards_lagged_1,total_points_lagged_1,penalties_saved_lagged_1,own_goals_lagged_1,xg_involvements_per_90_lagged_1,team_a_score_lagged_1,goals_scored_lagged_1,bonus_lagged_1,goals_conceded_lagged_3,player_match_xa_expected_lagged_3,transfers_trend_lagged_3,clean_sheets_lagged_3,transfers_in_lagged_3,ict_index_lagged_3,xa_lagged_3,assists_lagged_3,player_match_xg_expected_lagged_3,hype_vs_odds_ratio_lagged_3,xg_lagged_3,value_lagged_3,starts_lagged_3,xg_per_90_lagged_3,transfers_out_lagged_3,xa_per_90_lagged_3,xg_involvements_lagged_3,transfers_balance_lagged_3,team_h_score_lagged_3,penalties_missed_lagged_3,red_cards_lagged_3,xg_conceded_per_90_lagged_3,yellow_cards_lagged_3,selected_lagged_3,total_points_lagged_3,penalties_saved_lagged_3,saves_lagged_3,threat_lagged_3,xg_conceded_lagged_3,own_goals_lagged_3,xg_involvements_per_90_lagged_3,minutes_lagged_3,team_a_score_lagged_3,match_value_efficiency_lagged_3,goals_scored_lagged_3,creativity_lagged_3,xp_lagged_3,influence_lagged_3,bonus_lagged_3,goals_conceded_lagged_5,player_match_xa_expected_lagged_5,transfers_trend_lagged_5,clean_sheets_lagged_5,transfers_in_lagged_5,ict_index_lagged_5,xa_lagged_5,assists_lagged_5,player_match_xg_expected_lagged_5,hype_vs_odds_ratio_lagged_5,xg_lagged_5,value_lagged_5,starts_lagged_5,xg_per_90_lagged_5,transfers_out_lagged_5,xa_per_90_lagged_5,xg_involvements_lagged_5,transfers_balance_lagged_5,team_h_score_lagged_5,penalties_missed_lagged_5,red_cards_lagged_5,xg_conceded_per_90_lagged_5,yellow_cards_lagged_5,selected_lagged_5,total_points_lagged_5,penalties_saved_lagged_5,saves_lagged_5,threat_lagged_5,xg_conceded_lagged_5,own_goals_lagged_5,xg_involvements_per_90_lagged_5,minutes_lagged_5,team_a_score_lagged_5,match_value_efficiency_lagged_5,goals_scored_lagged_5,creativity_lagged_5,xp_lagged_5,influence_lagged_5,bonus_lagged_5,goals_conceded_std_3,player_match_xa_expected_std_3,transfers_trend_std_3,clean_sheets_std_3,transfers_in_std_3,ict_index_std_3,xa_std_3,assists_std_3,player_match_xg_expected_std_3,hype_vs_odds_ratio_std_3,xg_std_3,value_std_3,starts_std_3,xg_per_90_std_3,transfers_out_std_3,xa_per_90_std_3,transfers_balance_std_3,team_h_score_std_3,penalties_missed_std_3,red_cards_std_3,xg_conceded_per_90_std_3,yellow_cards_std_3,selected_std_3,total_points_std_3,penalties_saved_std_3,saves_std_3,threat_std_3,xg_conceded_std_3,own_goals_std_3,xg_involvements_per_90_std_3,minutes_std_3,team_a_score_std_3,match_value_efficiency_std_3,goals_scored_std_3,creativity_std_3,xp_std_3,influence_std_3,bonus_std_3,goals_conceded_std_5,player_match_xa_expected_std_5,transfers_trend_std_5,clean_sheets_std_5,transfers_in_std_5,ict_index_std_5,xa_std_5,assists_std_5,player_match_xg_expected_std_5,hype_vs_odds_ratio_std_5,xg_std_5,value_std_5,starts_std_5,xg_per_90_std_5,transfers_out_std_5,xa_per_90_s

In [33]:
cols_map = FeaturesConfig()


In [34]:
num_cols = [c for c in data.columns if c not in cols_map.id_cols + cols_map.static_cols + cols_map.target]
cat_cols = cols_map.static_cols

In [35]:
data.shape[1], len(num_cols + cat_cols + cols_map.id_cols)

(227, 226)

In [36]:
fpl_data_pipe = FPLDataPipe(num_cols=num_cols, 
                            cat_cols=cat_cols)


In [ ]:
class PipelineState(NamedTuple):
    data_splits: dict
    preprocessor: Any

@dataclass
class ModelConfig:
    model: Any
    param_grid: Dict[str, Any]

@dataclass
class ParamGrid:
    glm: Dict[str, Any]
    xgboost: Dict[str, Any]
    catboost: Optional[Dict[str, Any]] = None

device = device

param_grid = ParamGrid(
    glm={
    'alpha': stats.loguniform(1e-5, 0.01),
    'l1_ratio': stats.uniform(loc=0.0, scale=1.0),
    "family" : ['poisson', 'negative.binomial', 'tweedie', "normal"],
    "link" : ["auto", "log"]
    
    },
    # catboost={
    #     "iterations": stats.randint(150, 500),
    #     "depth": stats.randint(4, 7), 
    #     "learning_rate": stats.uniform(0.01, 0.05),
    #     "l2_leaf_reg": stats.uniform(1.0, 8.0),
    #     "bagging_temperature": stats.uniform(0.0, 5.0),
    #     "random_strength": stats.uniform(0.5, 5.0),
    #     "border_count": stats.randint(32, 255),
    # },
    xgboost={
        "n_estimators": stats.randint(150, 500),
        "max_depth": stats.randint(3, 7),
        "learning_rate": stats.uniform(0.01, 0.05),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.6, 0.3),
        "min_child_weight": stats.randint(1, 10),
        "gamma": stats.uniform(0.0, 5.0),
        "reg_alpha": stats.uniform(0.0, 2.0),
        "reg_lambda": stats.uniform(0.5, 3.0),
    }
    
)

model_registry = {
    "glm": ModelConfig(
        model=glum.GeneralizedLinearRegressor(
            random_state=77),
        param_grid=param_grid.glm
    ),
    # "catboost": ModelConfig(
    #     model=catboost.CatBoostRegressor(
    #         random_state=77, 
    #         loss_function="RMSE", 
    #         task_type="GPU" if device=="cuda" else "CPU",
    #         verbose=0,
    #         allow_writing_files=False
    #     ),
    #     param_grid=param_grid.catboost
    # ),
    "xgboost": ModelConfig(
        model=xgboost.XGBRegressor(
            random_state=77, 
            objective='reg:squarederror', 
            eval_metric='rmse', 
            device=device,
            enable_categorical=True,
              
        ),  
        param_grid=param_grid.xgboost
    )
    
}

In [38]:
data_splits = train_test_split(data)
pipeline_state = PipelineState(data_splits=data_splits,
                            preprocessor=fpl_data_pipe.preprocessor)

In [39]:
def calculate_scaled_bounds(df: pd.DataFrame, value_idx: int) -> tuple[float, float]:
    prices = df.iloc[:, value_idx].to_numpy()
    
    return float(np.quantile(prices, 0.50)), float(np.quantile(prices, 0.955))

In [40]:
def split_xy(pipeline_state: PipelineState):
    data_splits, preprocessor = pipeline_state.data_splits, pipeline_state.preprocessor
    X_train = preprocessor.fit_transform(data_splits["X_train"]).drop(columns=["date"])
    X_valid = preprocessor.transform(data_splits["X_valid"]).drop(columns=["date"])
    X_test  = preprocessor.transform(data_splits["X_test"]).drop(columns=["date"])

    y_train = data_splits["y_train"]
    y_valid = data_splits["y_valid"]
    y_test  = data_splits["y_test"]

    return (X_train, X_valid, X_test, y_train, y_valid, y_test)



In [41]:
def train_models(model_registry: dict, pipeline_state: PipelineState):
    
    data_splits = pipeline_state.data_splits

    results_list = []
    params = {}

    for name, cfg in model_registry.items():
        model = cfg.model
        grid = cfg.param_grid

        X_train = data_splits.get("X_train")
        y_train = data_splits.get("y_train")

        # if name == "catboost":
        #     fpl_pipe_cb = FPLDataPipe(num_cols=num_cols, cat_cols=cat_cols, catboost_flag=True)
        #     X_train = fpl_pipe_cb.preprocessor.fit_transform(X_train)
        #     
        #     cat_features = [col for col in X_train.columns if col.startswith("col__")]
        #     X_train[cat_features] = X_train[cat_features].astype(str)
        # else:
        fpl_pipe_xgb = FPLDataPipe(num_cols=num_cols, cat_cols=cat_cols, catboost_flag=False)
        X_train = fpl_pipe_xgb.preprocessor.fit_transform(X_train)
        cat_features = None

        val_col_name = [col for col in X_train.columns if "value" in col][0]
        value_idx = X_train.columns.get_loc(val_col_name)

        p_low, p_high = calculate_scaled_bounds(df=X_train, value_idx=value_idx)
        
        selector = ModelSelector(
            random_state=77, 
            p_low=p_low, 
            p_high=p_high, 
            value_idx=value_idx
        )
        
        

        print(f"Model: {name} | shape: {X_train.shape} | val_idx: {value_idx}", flush=True)
        
        df_results, best_models_map = selector.params_search(
            models=[model],
            models_names=[name],
            params_grid=[grid], 
            X_train=X_train,
            y_train=y_train,
            cv=5,
            n_iter=30,
            scoring="neg_mean_squared_error",
            cat_features=cat_features
        )

        results_list.append(df_results)
        params[name] = best_models_map[name]

    return pd.concat(results_list, axis=0), params

In [ ]:
results, params = train_models(
    model_registry=model_registry, 
    pipeline_state=pipeline_state)

Model: glm | shape: (80618, 220) | val_idx: 18
Fitting 5 folds for each of 30 candidates, totalling 150 fits
Model: xgboost | shape: (80618, 220) | val_idx: 18
Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [ ]:
results

,data,name,cv_mean_neg_mean_squared_error,best_params,mse@budget,mse@mid,mse@premium,mae@budget,mae@mid,mae@premium,r2
0,train,glm,-5.305475,"{'alpha': 91.91190317991818, 'family': 'poisso...",2.894968,5.709267,20.842873,1.240148,1.473125,2.814708,-1.192093e-07
0,train,xgboost,-3.562025,"{'colsample_bytree': 0.7822410302262526, 'gamm...",1.706366,3.463832,9.031380,0.585367,1.025187,2.184565,4.271091e-01


In [ ]:
fpl_pipe_cb = FPLDataPipe(num_cols=num_cols, 
                          cat_cols=cat_cols, 
                          catboost_flag=False)

fpl_pipe_cb.preprocessor.fit(data_splits.get("X_train"))
X_valid = fpl_pipe_cb.preprocessor.transform(data_splits.get("X_valid"))

In [ ]:
importances = pd.DataFrame({"importance": params.get("xgboost").feature_importances_, "features": X_valid.columns})

In [ ]:
importances.sort_values(by="importance", ascending=False).head(20)

,importance,features
12,0.138127,num__ict_index_ema_3
30,0.126731,num__total_points_ema_3
60,0.029466,num__total_points_lagged_1
29,0.014062,num__selected_ema_3
41,0.009337,num__xp_ema_3
18,0.008391,num__value_ema_3
43,0.008068,num__goals_conceded_lagged_1
46,0.007802,num__clean_sheets_lagged_1
150,0.007674,num__ict_index_std_3
38,0.006239,num__match_value_efficiency_ema_3


In [ ]:
coefs = pd.DataFrame({"coefs": params.get("glm").coef_, "features": X_valid.columns})
coefs.coefs.unique()

array([0.])

In [ ]:
X_test = fpl_pipe_cb.preprocessor.transform(data_splits.get("X_test"))

In [ ]:
preds_valid_ens = pd.DataFrame({
    "glm": params.get("glm").predict(X_valid).clip(0, 15),
    "xgboost": params.get("xgboost").predict(X_valid).clip(0, 15),
})
if config.ML_PREDS_PATH.exists():
    existing_valid = pd.read_parquet(config.ML_PREDS_PATH)
    if len(existing_valid) == len(preds_valid_ens):
        for col in preds_valid_ens.columns:
            existing_valid[col] = preds_valid_ens[col].values
        preds_valid_ens = existing_valid
preds_valid_ens.to_parquet(config.ML_PREDS_PATH)
print(f"Preds valid shape: {preds_valid_ens.shape} -> {config.ML_PREDS_PATH}")


In [ ]:
meta_cols = ["name", "code", "team", "opponent_team", "position", "was_home", "season", "gw", "kickoff_time"]
X_test_raw = data_splits["X_test"]
X_predict_raw = data_splits["X_predict"]

test_meta = X_test_raw[meta_cols].copy().reset_index(drop=True)
test_meta["is_played"] = True

if not X_predict_raw.empty:
    X_predict = fpl_pipe_cb.preprocessor.transform(X_predict_raw)
    pred_meta = X_predict_raw[meta_cols].copy().reset_index(drop=True)
    pred_meta["is_played"] = False
    season_2627_meta = pd.concat([test_meta, pred_meta], ignore_index=True)
else:
    X_predict = None
    season_2627_meta = test_meta

for algo_name in ["glm", "xgboost"]:
    algo_df = season_2627_meta.copy()
    p_test = params.get(algo_name).predict(X_test).clip(0, 15)
    if X_predict is not None:
        p_next = params.get(algo_name).predict(X_predict).clip(0, 15)
        algo_df[algo_name] = np.concatenate([p_test, p_next])
    else:
        algo_df[algo_name] = p_test
    target_path = config.get_preds_path(algo_name)
    algo_df.to_parquet(target_path, index=False)
    print(f"Preds 26/27 ({algo_name}) shape: {algo_df.shape} -> {target_path}")
